# Campaign dynamics and observable/oracle graph investigation

**Goal.** Follow evolving fraud campaigns through state transitions, temporal graph construction, and an as-of investigator view. Compare observable evidence with oracle truth without leaking future events.

**Audience.** Data/ML scientists, fraud analysts, platform engineers, and researchers.

**Prerequisites.** Python 3.12+, a clean checkout, and the base FraudTwin install. The workflow is deterministic and runs offline; service integrations are deliberately out of scope here.

**Source size.** 1,000–10,000 logical payments. Every section writes only compact summaries, manifests, or fingerprints to a temporary directory.

**Interpretation.** Synthetic evidence demonstrates mechanics and invariants, not production prevalence or model performance guarantees.


## Campaign dynamics and evolving fraud rings



In [ ]:
from collections import Counter
from pathlib import Path

from fraudtwin.config import load_config
from fraudtwin.generation import generate
from fraudtwin.reproducibility import sha256_json

config = load_config(Path("configs/benchmarks/m15-campaign-dynamics-v1.yaml"))
data = generate(config, write=False)
dynamic = data.behavior.campaign_dynamics
print(
    {
        "run_id": data.run_id,
        "payments": len(data.behavior.payments),
        "campaigns": len(data.behavior.graph_campaigns),
    }
)
assert dynamic is not None and dynamic.active

In [ ]:
print(
    {
        "snapshots": len(dynamic.snapshots),
        "transitions": len(dynamic.transitions),
        "phase_changes": len(dynamic.phase_changes),
        "membership_changes": len(dynamic.membership_changes),
        "intensity": len(dynamic.intensity_decisions),
    }
)
assert dynamic.snapshots and dynamic.transitions

In [ ]:
campaigns = [
    {"id": x.campaign_id, "scenario": x.scenario_type, "participants": len(x.participant_ids)}
    for x in data.behavior.graph_campaigns
]
print(campaigns)
assert len(campaigns) == 3

In [ ]:
phases = Counter(x.phase for x in dynamic.snapshots)
print({"phase_counts": dict(sorted(phases.items()))})
assert phases

In [ ]:
transitions = [
    {"campaign": x.campaign_id, "from": x.from_phase, "to": x.to_phase} for x in dynamic.transitions
]
print(transitions[:8])
assert transitions

In [ ]:
members = Counter(x.campaign_id for x in dynamic.membership_changes)
print({"membership_changes": dict(members)})
assert members

In [ ]:
lineage = [
    {
        "campaign": x.campaign_id,
        "derived_events": len(x.derived_event_ids),
        "derived_payments": len(x.derived_payment_ids),
    }
    for x in dynamic.lineage
]
print(lineage)
assert all(x["derived_events"] >= 0 for x in lineage)

In [ ]:
times = [x.captured_at for x in dynamic.source_snapshots]
print({"source_snapshots": len(times), "first": min(times), "last": max(times)})
assert len(times) == len(dynamic.source_snapshots)

In [ ]:
manifest = {
    "run_id": data.run_id,
    "configuration_hash": dynamic.configuration_hash,
    "streams": dynamic.stream_ids,
    "campaigns": campaigns,
    "transitions": len(dynamic.transitions),
}
manifest["fingerprint"] = sha256_json(manifest)
print(manifest)

In [ ]:
assert manifest["fingerprint"] == sha256_json(
    {k: v for k, v in manifest.items() if k != "fingerprint"}
)
print("Campaign state changes are linked to phases, actors, and source lineage.")

## Observable versus oracle investigation



In [ ]:
from datetime import timedelta
from pathlib import Path

from fraudtwin.config import load_config
from fraudtwin.generation import generate
from fraudtwin.graph import build_graph
from fraudtwin.reproducibility import sha256_json

config = load_config(Path("configs/benchmarks/m11-graph-v2.yaml"))
data = generate(config, write=False)
observable = build_graph(config, data.entities, data.behavior, data.manifest, view="observable")
oracle = build_graph(config, data.entities, data.behavior, data.manifest, view="oracle")
print(
    {
        "run_id": data.run_id,
        "observable_edges": len(observable.edges),
        "oracle_edges": len(oracle.edges),
    }
)
assert len(oracle.edges) >= len(observable.edges)

In [ ]:
print(
    {
        "nodes": {"observable": len(observable.nodes), "oracle": len(oracle.nodes)},
        "edges": {"observable": len(observable.edges), "oracle": len(oracle.edges)},
        "patterns": len(observable.patterns),
    }
)
assert len(oracle.nodes) >= len(observable.nodes)

In [ ]:
cutoff = config.simulation.start + timedelta(days=2)
observed_asof = build_graph(
    config, data.entities, data.behavior, data.manifest, view="observable", as_of=cutoff
)
oracle_asof = build_graph(
    config, data.entities, data.behavior, data.manifest, view="oracle", as_of=cutoff
)
print(
    {
        "cutoff": cutoff.isoformat(),
        "observed_edges": len(observed_asof.edges),
        "oracle_edges": len(oracle_asof.edges),
    }
)
assert len(observed_asof.edges) <= len(observable.edges)

In [ ]:
available = [
    edge
    for edge in observed_asof.edges
    if edge.available_at is not None and edge.available_at <= cutoff
]
print(
    {"available_by_cutoff": len(available), "excluded": len(observed_asof.edges) - len(available)}
)
assert len(available) <= len(observed_asof.edges)

In [ ]:
oracle_only = len(oracle_asof.edges) - len(observed_asof.edges)
print({"oracle_only_edges": oracle_only, "meaning": "latent truth is not investigator-visible"})
assert oracle_only >= 0

In [ ]:
print([pattern.pattern_type for pattern in observable.patterns[:5]])
print({"observable_patterns": len(observable.patterns)})

In [ ]:
campaign_types = {}
for campaign in observable.campaigns:
    campaign_types[campaign.scenario_type] = campaign_types.get(campaign.scenario_type, 0) + 1
print(campaign_types)
assert isinstance(campaign_types, dict)

In [ ]:
post_cutoff = [
    edge for edge in observed_asof.edges if edge.event_time is not None and edge.event_time > cutoff
]
print({"post_cutoff_edges": len(post_cutoff)})
assert not post_cutoff

In [ ]:
manifest = {
    "run_id": data.run_id,
    "cutoff": cutoff.isoformat(),
    "observable_edges": len(observable.edges),
    "oracle_edges": len(oracle.edges),
    "oracle_only_at_cutoff": oracle_only,
}
manifest["fingerprint"] = sha256_json(manifest)
print(manifest)

In [ ]:
assert manifest["fingerprint"] == sha256_json(
    {k: v for k, v in manifest.items() if k != "fingerprint"}
)
print(
    "Observable graphs support leakage-safe investigation; oracle graphs explain benchmark truth."
)

## Verification and next step

Re-run the offline cells from a clean checkout and compare the printed fingerprints. For service-backed publication, continue with the relevant integration runbook after this notebook; do not treat synthetic metrics as a deployment SLO.
